In [184]:
import numpy as np
import pandas as pd


In [185]:
df = pd.read_csv("campusvolt_energy_data.csv")

In [186]:
df["timestamp"].dtype

<StringDtype(storage='python', na_value=nan)>

In [187]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43800 entries, 0 to 43799
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   timestamp         43800 non-null  str    
 1   meterId           43800 non-null  str    
 2   buildingId        43800 non-null  str    
 3   buildingName      43800 non-null  str    
 4   voltage           43800 non-null  float64
 5   current           43800 non-null  float64
 6   power             43800 non-null  float64
 7   energy            43800 non-null  float64
 8   frequency         43800 non-null  float64
 9   powerFactor       43800 non-null  float64
 10  temperature       43800 non-null  float64
 11  occupancyLevel    43800 non-null  float64
 12  weatherCondition  43800 non-null  str    
 13  dayType           43800 non-null  str    
 14  workingHours      43800 non-null  bool   
 15  scenario          43800 non-null  str    
 16  status            43800 non-null  str    
 17  simu

In [188]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [189]:
df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["dayOfWeek"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month

In [190]:
df.groupby("meterId")
df.groupby("timestamp")

In [191]:
df.head()

,timestamp,meterId,buildingId,buildingName,voltage,current,power,energy,frequency,powerFactor,...,dayType,workingHours,scenario,status,simulationTime,simulationSpeed,hour,day,dayOfWeek,month
0,2025-01-01 00:00:00,M001,B001,Academic Block,232.30,53.68,19.37,19.37,49.962,0.891,...,Weekday,False,Normal,Normal,2025-01-01 00:00:00,1,0,1,2,1
1,2025-01-01 01:00:00,M001,B001,Academic Block,226.96,44.72,15.56,34.93,50.025,0.866,...,Weekday,False,Normal,Normal,2025-01-01 01:00:00,1,1,1,2,1
2,2025-01-01 02:00:00,M001,B001,Academic Block,230.33,33.41,11.96,46.89,49.908,0.889,...,Weekday,False,Normal,Normal,2025-01-01 02:00:00,1,2,1,2,1
3,2025-01-01 03:00:00,M001,B001,Academic Block,232.47,31.70,11.61,58.50,49.902,0.883,...,Weekday,False,Normal,Normal,2025-01-01 03:00:00,1,3,1,2,1
4,2025-01-01 04:00:00,M001,B001,Academic Block,229.10,38.65,13.48,71.98,49.882,0.873,...,Weekday,False,Normal,Normal,2025-01-01 04:00:00,1,4,1,2,1


In [192]:
df["energyConsumption"] = df.groupby("meterId")["energy"].diff()

In [193]:
df["energyConsumption"].isnull().sum()

np.int64(5)

In [194]:
df = df.dropna()

In [195]:
df["energyConsumption"].isnull()

1        False
2        False
3        False
4        False
5        False
         ...  
43795    False
43796    False
43797    False
43798    False
43799    False
Name: energyConsumption, Length: 43795, dtype: bool

In [196]:
features = ["voltage", "current", "power", "frequency", "powerFactor",
"temperature", "occupancyLevel", "workingHours",
"hour", "dayOfWeek", "month",
"meterId", "weatherCondition", "dayType"]

In [197]:
set(features) - set(df.columns)

set()

In [198]:
X = df[features]
y = df["energyConsumption"]

In [199]:
X.shape
y.shape

(43795,)

In [200]:
print(df["meterId"].unique())
print(df["weatherCondition"].unique())
print(df["dayType"].unique())


<StringArray>
['M001', 'M002', 'M003', 'M004', 'M005']
Length: 5, dtype: str
<StringArray>
['Sunny', 'Cloudy', 'Rainy', 'Stormy']
Length: 4, dtype: str
<StringArray>
['Weekday', 'Weekend']
Length: 2, dtype: str


In [201]:
X = pd.get_dummies(
    X,
    columns=["meterId", "weatherCondition", "dayType"]
)

In [206]:
print(X)

       voltage  current  power  frequency  powerFactor  temperature  \
1       226.96    44.72  15.56     50.025        0.866        20.73   
2       230.33    33.41  11.96     49.908        0.889        20.94   
3       232.47    31.70  11.61     49.902        0.883        19.98   
4       229.10    38.65  13.48     49.882        0.873        21.42   
5       227.97    44.77  15.61     50.049        0.895        25.39   
...        ...      ...    ...        ...          ...          ...   
43795   228.87    41.80  14.64     49.974        0.885        19.42   
43796   225.27    51.25  17.62     49.916        0.883        19.88   
43797   229.64    20.40   7.02     49.981        0.890        18.20   
43798   228.80    34.80  12.33     49.935        0.884        22.71   
43799   230.68    38.37  13.77     49.932        0.900        22.10   

       occupancyLevel  workingHours  hour  dayOfWeek  ...  meterId_M002  \
1                0.00         False     1          2  ...         False 

In [202]:
X.shape

(43795, 22)

In [203]:
X.dtypes

voltage                    float64
current                    float64
power                      float64
frequency                  float64
powerFactor                float64
temperature                float64
occupancyLevel             float64
workingHours                  bool
hour                         int32
dayOfWeek                    int32
month                        int32
meterId_M001                  bool
meterId_M002                  bool
meterId_M003                  bool
meterId_M004                  bool
meterId_M005                  bool
weatherCondition_Cloudy       bool
weatherCondition_Rainy        bool
weatherCondition_Stormy       bool
weatherCondition_Sunny        bool
dayType_Weekday               bool
dayType_Weekend               bool
dtype: object

In [204]:
df["meterId"].value_counts()

meterId
M001    8759
M002    8759
M003    8759
M004    8759
M005    8759
Name: count, dtype: int64

In [205]:
split_point = int(8759 * 0.8)
print(split_point)

7007


In [207]:
train_indices = []
test_indices = []

for meter, group in df.groupby("meterId"):
    
    train_indices.extend(group.index[:7007])
    test_indices.extend(group.index[7007:])

In [208]:
X_train = X.loc[train_indices]
X_test = X.loc[test_indices]

y_train = y.loc[train_indices]
y_test = y.loc[test_indices]

In [209]:
X_train.shape
X_test.shape

y_train.shape
y_test.shape

(8760,)